<a href="https://colab.research.google.com/github/JoseAlberto88/Hugging-Face-Text-Classification/blob/main/huggingface_text_classification_tutorial_video.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Learning Hugging Face Text Classification Tutorial

* Resources notebook: https://www.learnhuggingface.com/notebooks/hugging_face_text_classification_tutorial
* Setup steps: https://www.learnhuggingface.com/extras/setup

**Note** A GPU is needed on Google Colab, go to Runtime -> Change runtime -> Hardware accelerator -> GPU.

### Import necessary libraries

In [1]:
import transformers

In [2]:
# Install dependencies (this is mostly for Google Colab)
try:
  import datasets, evaluate, accelerate
  import gradio as gr
except ModuleNotFoundError:
  !pip install -U datasets evaluate accelerate gradio
  import datasets, evaluate, accelerate
  import gradio as gr

import random

import numpy as np
import pandas as pd

import torch
import transformers

print(f"Using transformers version: {transformers.__version__}")
print(f"Using torch version: {torch.__version__}" )
print(f"Using datasets version: {datasets.__version__}")


Using transformers version: 5.15.0
Using torch version: 2.11.0+cpu
Using datasets version: 5.0.1


## 3. Getting a dataset

Building food not food text classification model: need food not food text dataset.

In [3]:
from datasets import load_dataset

dataset = load_dataset(path="mrdbourke/learn_hf_food_not_food_image_captions")
dataset

README.md:   0%|          | 0.00/1.32k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 11.9kB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/250 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 250
    })
})

In [4]:
# What features are there ?

dataset.column_names

{'train': ['text', 'label']}

In [5]:
dataset["train"]

Dataset({
    features: ['text', 'label'],
    num_rows: 250
})

In [6]:
dataset["train"][0]

{'text': 'Creamy cauliflower curry with garlic naan, featuring tender cauliflower in a rich sauce with cream and spices, served with garlic naan bread.',
 'label': 'food'}

### Import random samples


In [7]:
import random

random_indexs = random.sample(range(len(dataset["train"])), 5)

random_samples = dataset["train"][random_indexs]

print(f"[INFO] Random samples from dataset:\n")
for text, label in zip(random_samples["text"], random_samples["label"]):
  print(f"Text: {text} | Label: {label}")

[INFO] Random samples from dataset:

Text: Swimming pool sparkling in a backyard | Label: not_food
Text: Bowl of sashimi with thin slices of raw fish. | Label: food
Text: Family gathered around a dining table, laughing and eating | Label: not_food
Text: Pizza with a seasonal theme, featuring toppings like butternut squash and kale | Label: food
Text: Set of stainless steel utensils arranged on a kitchen table | Label: not_food


In [8]:
# Get unique label values
dataset["train"].unique("label")

['food', 'not_food']

In [9]:
# Check the count of each label
from collections import Counter

Counter(dataset["train"]["label"])

Counter({'food': 125, 'not_food': 125})

In [10]:
# Turn our dataset into a DataFrame and get a random sample
food_not_food_df = pd.DataFrame(dataset["train"])
food_not_food_df.sample(7)

,text,label
76,Set of bowls stacked on a shelf,not_food
0,"Creamy cauliflower curry with garlic naan, fea...",food
41,"Brussels sprouts in a bowl, sprinkled with bac...",food
228,A close-up of a cat lounging on a windowsill w...,not_food
101,Remote control placed on a couch cushion,not_food
114,Three black dogs laying on the garage floor wi...,not_food
209,A close-up of a man and his dog sharing a quie...,not_food


In [11]:
food_not_food_df["label"].value_counts()

,count
label,
food,125
not_food,125


## 4. Preparing data for text classification

We want to:

1. Tokenize our text -> turn our text into numbers (this goes for labels as well).
2. Create a train/test split -> want to train our model on the training split and want to evaluate our model on the test split.

In [12]:
# Create a mapping for labels to numeric value

id2label = {0: "not_food", 1: "food"}
label2id = {"not_food" : 0, "food" : 1}

print(id2label)
print(label2id)

{0: 'not_food', 1: 'food'}
{'not_food': 0, 'food': 1}


In [13]:
# Create mappings programmatically from dataset
id2label = {idx : label for idx, label in enumerate(dataset["train"].unique("label")[::-1])}
id2label

{0: 'not_food', 1: 'food'}

In [14]:
label2id = {label : idx for idx, label in enumerate(dataset["train"].unique("label")[::-1])}
label2id

{'not_food': 0, 'food': 1}

In [15]:
# turn labels into 0 or 1

def map_labels_to_number(example):
  example["label"] = label2id[example["label"]]
  return example

example_sample = {"text" : "This is a sentence about my favprite food: honey", "label": "food"}

# Test our function
map_labels_to_number(example_sample)

{'text': 'This is a sentence about my favprite food: honey', 'label': 1}

In [16]:
# Map our dataset labels to numbers (the whole thing)
# We do this with dataset.map()  - https://huggingface.co/docs/datasets/process#map
dataset = dataset["train"].map(map_labels_to_number)
dataset[:5]

Map:   0%|          | 0/250 [00:00<?, ? examples/s]

{'text': ['Creamy cauliflower curry with garlic naan, featuring tender cauliflower in a rich sauce with cream and spices, served with garlic naan bread.',
  'Set of books stacked on a desk',
  'Watching TV together, a family has their dog stretched out on the floor',
  'Wooden dresser with a mirror reflecting the room',
  'Lawn mower stored in a shed'],
 'label': [1, 0, 0, 0, 0]}

In [17]:
# Shuffle data and look at more 5 random examples
dataset.shuffle()[:5]

{'text': ['Three black dogs laying on the garage floor with a blue car in the background',
  'Spicy prawn curry with fresh mint garnish, featuring juicy prawns in a fiery sauce with onions and tomatoes, finished with mint leaves.',
  'Garage door with a remote control ready for use',
  'Low-carb sushi roll with cucumber or seaweed wraps instead of rice.',
  'Treadmill available in a home gym'],
 'label': [0, 1, 0, 1, 0]}

### Split the dataset into training and test sets

* Train set = model will learn patters on this dataset
* Validation set (optional) = we can tune our model's hyperparameters on this set
* Test set = model will evaluate patters on this dataset

We can split our dataset using `datasets.Dataset.train_test_split`. https://huggingface.co/docs/datasets/v4.8.4/process#split

In [18]:
# Split our dataset into train/test splits

dataset = dataset.train_test_split(test_size=0.2, seed=42)
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 200
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 50
    })
})

In [19]:
random_idx_train = random.randint(0, len(dataset["train"]))
random_sample_train = dataset["train"][random_idx_train]
random_sample_train

{'text': 'Nigiri sushi topped with fresh salmon and tuna slices on vinegared rice.',
 'label': 1}

In [20]:
random_idx_test = random.randint(0, len(dataset["test"]))
random_sample_test = dataset["test"][random_idx_test]
random_sample_test

{'text': 'Luxurious coconut shrimp curry on a generous plate, featuring succulent shrimp in a rich coconut milk sauce, served with jasmine rice.',
 'label': 1}

### Tokenizing our text data (turning text into numbers)

The premise of tokenization is to turn words into numbers.

e.g. "I love pizza!" -> [30, 145, 678, 999]

-

The `transformers` library has in-built support for Hugging Face `tokenizers`.
And the class `transformers.AutoTokenizer` helps pair a model to a tokenizer.

-

* To find all models: https://huggingface.co/models
* Model/tokenizer we're going to use: https://huggingface.co/distilbert/distilbert-base-uncased
* Models are aften paired with tokenizers
* Tokenizer = turn text into numbers
* Model = finds patterns in those numbers

In [21]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(pretrained_model_name_or_path="distilbert-base-uncased",
                                          use_fast=True) # use the fast implementation (on by default, note:this requires Rust installed)
tokenizer


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

BertTokenizer(name_or_path='distilbert-base-uncased', vocab_size=30522, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
})

In [22]:
# Test out the tokenizer
tokenizer("I love pizza")

{'input_ids': [101, 1045, 2293, 10733, 102], 'token_type_ids': [0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1]}

* `input_ids` = our text turned into numbers
* `attention_mask` = whether or not to pay attention to certain tokens (1 = yes pay attention, 0 = no don't pay attention)



In [23]:
# Get the lenght of our tokenizer vocab
length_of_tokenizer_vocab = len(tokenizer.vocab)
print(f"[INFO] Number of items in our tokenizer vocab: {length_of_tokenizer_vocab}")

# Get the maximum sequence lenght the tokenizer can handle
max_tokenizer_input_sequence_length = tokenizer.model_max_length
print(f"[INFO] Max tokenizer input sequence length: {max_tokenizer_input_sequence_length}")

[INFO] Number of items in our tokenizer vocab: 30522
[INFO] Max tokenizer input sequence length: 512


In [24]:
# Does "daniel" occur in the vocab?
tokenizer.vocab["daniel"]

3817

In [25]:
tokenizer.vocab["x"]

1060

In [26]:
tokenizer("akash")

{'input_ids': [101, 9875, 4095, 102], 'token_type_ids': [0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1]}

In [27]:
tokenizer.convert_ids_to_tokens(tokenizer("akash").input_ids)

['[CLS]', 'aka', '##sh', '[SEP]']

In [28]:
# Try to tokenize an emoji
tokenizer.convert_ids_to_tokens(tokenizer("pizza").input_ids)

['[CLS]', 'pizza', '[SEP]']

In [29]:
# Get the first 5 items in the tokenizer vocab
sorted(tokenizer.vocab.items())[:5]

[('!', 999), ('"', 1000), ('#', 1001), ('##!', 29612), ('##"', 29613)]

In [31]:
import random

random.sample(sorted(tokenizer.vocab.items()), k=5)

[('pained', 22295),
 ('disqualification', 27782),
 ('##pm', 9737),
 ('privileges', 14310),
 ('1835', 10150)]